# TrappyTV Examplar notebook

In [1]:
# Import Packages
import os
import sys
import numpy as np
import pandas as pd

In [2]:
## Import Bokeh module and TrappyTV class
from bokeh.io import output_notebook
output_notebook()

import sys
sys.path.append(os.path.dirname(os.getcwd()))
from trappytv.trappytv2 import TrappyTV2 as TrappyTV

Loading BokehJS ...

## Import Data

In [3]:
datapaths = ["../data/merged_tracks.hd5"]
#datapaths = ["data/2026_01_02/m2_merged_tracks.hd5"]
#datapaths = ["data/2025_12_30/M4_2025_12_30.hd5"]

In [4]:
class CellView: ## Datastructure to import 
    def __init__(self, datapath):
        if datapath.endswith(".hd5"):
            print("HDF datastore mode!")
            self.scopeid = "Trappy-Scope"
            self.paths = {"tracks": datapath}
        else:
            self.scopeid = os.path.basename(datapath)[:2]
            self.postprocess_path = os.path.join(datapath, "postprocess")
            self.paths = {"tracks": os.path.join(self.postprocess_path, "merged_tracks.hd5"),
                        "xyr_df": os.path.join(self.postprocess_path, "xyr.hd5")
                        }
            self.first_frames = np.load(os.path.join(self.postprocess_path, "first_frames.npy"))
        
        self.dfs = {key: pd.read_hdf(value, key="df") for key, value in self.paths.items()}
        try:
            self.metadata = pd.read_hdf(self.paths["tracks"], key="metadata")
        except:
            print("[WARNING] Metadata not found!")

        try:
            self.dfs["xyr_df"] = pd.read_hdf(self.paths["tracks"], key="xyr_df")
        except Exception as e:
            print(e)
            print("[WARNING] XYR data msiing")
        
        if "speed" not in self.dfs["tracks"].columns:
            self.dfs["tracks"]["speed"] = np.hypot(np.gradient(self.dfs["tracks"]["x_unrefined"]), np.gradient(self.dfs["tracks"]["y_unrefined"]))
        self.stuff = {}
    def __call__(self):
        return self.dfs["tracks"]

In [5]:
cell = CellView(datapaths[0])
cell()

HDF datastore mode!
[WARNING] Metadata not found!
'No object named xyr_df in the file'
[WARNING] XYR data msiing


,relative_ms_timestamp,tpts_gap,frame,split,dt,gframe,dt_diff,elapsed_seconds,elapsed_frames,x,...,speed,xf,yf,speedf,xs_long,ys_long,vxs_long,vys_long,axs_long,ays_long
gframe,,,,,,,,,,,,,,,,,,,,,
0,0.000000e+00,False,0,0,2025-08-21 10:03:34.117535526,0,NaT,NaN,0,502.713453,...,NaN,502.823007,1335.304534,0.000000,502.044352,1335.986338,-37172.138303,-16266.583902,-7.674861e+06,-9.529556e+06
1,4.000000e+01,False,1,0,2025-08-21 10:03:34.157535526,1,0 days 00:00:00.040000,0.040000,1,502.418846,...,1.938684,500.989535,1334.674555,1.938684,500.439552,1335.278513,-37493.830651,-16662.928249,-7.292963e+06,-8.911704e+06
2,7.999500e+01,False,2,0,2025-08-21 10:03:34.197530529,2,0 days 00:00:00.039995003,0.039995,1,499.434843,...,2.422911,498.787465,1333.663918,2.422911,498.415060,1334.370879,-37872.794426,-17120.931305,-6.815678e+06,-8.139531e+06
3,1.199860e+02,False,3,0,2025-08-21 10:03:34.237521526,3,0 days 00:00:00.039990997,0.039991,1,496.592272,...,2.087437,496.705202,1333.517038,2.087437,496.655053,1333.569997,-38178.726410,-17482.255503,-6.404477e+06,-7.474272e+06
4,1.599860e+02,False,4,0,2025-08-21 10:03:34.277521518,4,0 days 00:00:00.039999992,0.040000,1,494.736997,...,1.981949,494.747139,1333.210257,1.981949,494.971277,1332.794747,-38451.585144,-17796.825048,-6.014057e+06,-6.842632e+06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1230520,4.921424e+07,False,15003,81,2025-08-21 23:43:48.357535526,1230520,0 days 00:00:00.040000,0.040000,1,113.493795,...,0.457332,113.919441,850.953409,0.457332,116.320308,851.489582,1535.294649,-321.504468,2.210757e+06,1.300971e+05
1230521,4.921428e+07,False,15004,81,2025-08-21 23:43:48.397535526,1230521,0 days 00:00:00.040000,0.040000,1,113.395311,...,0.459689,114.304509,850.702339,0.459689,116.376061,851.478281,1614.103188,-316.810680,2.240828e+06,1.350365e+05
1230522,4.921432e+07,False,15005,81,2025-08-21 23:43:48.437535526,1230522,0 days 00:00:00.040000,0.040000,1,114.309767,...,0.713439,114.377463,851.412038,0.713439,116.468165,851.461080,1738.523012,-309.179546,2.287497e+06,1.427026e+05


## View Data

In [6]:
#%%time
#TrappyTV(cell, width=1000, height=1000).view_all(sample=10)  ## Show the whole dataset but sample 10 points
tv = TrappyTV(cell, width=1000, height=1000)
tv.view_split(split_no=0)    ## Show only a particular split -> render all points

In [ ]:
import bokeh
bokeh.io.save(tv.layout, "output.html")